In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import datetime
import scipy.stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.dates as mdates
import matplotlib


In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    decimal_part = decimal_year - year
    start_of_year = pd.to_datetime([f'{y}-01-01' for y in year])
    seconds_in_year = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - start_of_year).total_seconds()
    seconds = decimal_part * seconds_in_year
    return start_of_year + pd.to_timedelta(seconds, unit='s')

In [ ]:
decimal_year_to_date(1961.1)

In [ ]:
df = pd.read_csv("NewPath1926262012.csv",header=None).drop(3, axis=1)
df.columns = ["decimal_year", "d13c","sea_level"]
df.drop_duplicates("decimal_year", keep="first", inplace=True)
df.sort_values("decimal_year", inplace=True)
df.reset_index(inplace=True, drop=True)


df['datetime'] = decimal_year_to_datetime(df['decimal_year'].values)
df.set_index("datetime", inplace=True)
#df.resample("6ME").interpolate(method="time")
df

In [ ]:
enso = pd.read_csv("../ENSO_Time_Series.csv", parse_dates=[0])
#enso = enso.set_index("Date")
enso['day_of_year'] = enso['Date'].dt.dayofyear
enso['is_leap'] = enso['Date'].dt.is_leap_year
enso['total_days'] = np.where(enso['is_leap'], 366, 365)
enso['decimal_year'] = enso['Date'].dt.year + (enso['day_of_year'] - 1) / enso['total_days']
enso

In [ ]:
enso_phase = pd.read_csv('../ENSO_Phase_Classification.csv')
enso_phase.columns=['Year','Phase']
enso_phase['Date'] = decimal_year_to_datetime(enso_phase['Year'])
#enso_phase.set_index('Date', inplace=True)
enso_phase['Phase'] = enso_phase['Phase'].str.replace('Warm Phase', 'W').str.replace('Cold Phase', 'C').str.replace('Neutral Phase','N')
enso_phase

In [ ]:
window = 30

cs = scipy.interpolate.CubicSpline(df["decimal_year"].values ,df["d13c"].values,  bc_type='natural')
xs = np.arange(1926, 2012, 0.5)
xs_date = decimal_year_to_datetime(xs)
ys = cs(xs)
df["MA_d13c"] = df["d13c"].rolling(window, center=True).mean()
df["MA_sealevel"] = df["sea_level"].rolling(window, center=True).mean()
df

In [ ]:
enso_phase

In [ ]:
phase_color = {'W':'red', 'C':'blue'}
enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<df.index.max().year)&(enso_phase['Phase'] != 'N')]


fig, axes = plt.subplots(2,1, figsize=(8,6), sharex=True, gridspec_kw={'hspace':0})

ax = axes[1]
ax.plot(df['decimal_year'], df["d13c"], ".",label="raw")
#ax.plot(xs, ys, label="spline")
ax.plot(df['decimal_year'], df["MA_d13c"], label="MA w={:}".format(window))
ax.vlines(2007.25, -10,10, "red", alpha=0.5 )
ax.vlines(1961.2, -10,10, "magenta", alpha=0.5)


ax.text(1962.3, -3.75, r'1961 $M_W$ 6.4', 
        bbox=dict(facecolor="w", edgecolor="magenta"))

ax.text(1993.5, -0.2, r'2007 $M_W$ 8.1',  
        bbox=dict(facecolor="w", edgecolor="red"))

ax.legend(loc='lower left')
ax.set_ylabel(r'$\delta ^{13}$C')


for i, index in enumerate(enso_phase_selected.index):
    
    df_iter = enso_phase.loc[index]
    df_iter_next = enso_phase.loc[index+1]

    rect = matplotlib.patches.Rectangle((df_iter['Year'],-5),
                                        1, 10, 
                                        color = phase_color[df_iter['Phase']], 
                                        edgecolor=None, alpha=0.1) 
    ax.add_patch(rect)

ax.set_ylim(-4,0.5)
ax.grid()

ax2 = axes[0]

ax2.plot(enso['decimal_year'], enso["ENSO"], color='gray', label='ENSO')


for i, index in enumerate(enso_phase_selected.index):
    
    df_iter = enso_phase.loc[index]
    df_iter_next = enso_phase.loc[index+1]

    rect = matplotlib.patches.Rectangle((df_iter['Year'],-5),
                                        1, 10, 
                                        color = phase_color[df_iter['Phase']], 
                                        edgecolor=None, alpha=0.1) 
    ax2.add_patch(rect)

ax2.set_ylim(-2,3)
ax2.set_xlim(1925,2012)
ax2.grid()
ax2.legend()
ax2.set_ylabel("ENSO Index")


fig.savefig("ENSO_plot.png")

In [ ]:
phase_color = {'W':'red', 'C':'blue'}
enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<df.index.max().year)&(enso_phase['Phase'] != 'N')]


fig, axes = plt.subplots(2,1, figsize=(8,6), sharex=True, gridspec_kw={'hspace':0})

ax = axes[1]
ax.plot(df['decimal_year'], df["sea_level"], ".",label="raw")
#ax.plot(xs, ys, label="spline")
ax.plot(df['decimal_year'], df["MA_sealevel"], label="MA w={:}".format(window))
ax.vlines(2007.25, -200,100, "red", alpha=0.5 )
ax.vlines(1961.2, -200,100, "magenta", alpha=0.5)


ax.text(1962.3, -90, r'1961 $M_W$ 6.4', 
        bbox=dict(facecolor="w", edgecolor="magenta"))

ax.text(1993.5, -0.2, r'2007 $M_W$ 8.1',  
        bbox=dict(facecolor="w", edgecolor="red"))

ax.legend(loc='lower left')
ax.set_ylabel("Relative Sea Level"+"\n"+"[cm]")


for i, index in enumerate(enso_phase_selected.index):
    
    df_iter = enso_phase.loc[index]
    df_iter_next = enso_phase.loc[index+1]

    rect = matplotlib.patches.Rectangle((df_iter['Year'],-200),
                                        1, 300, 
                                        color = phase_color[df_iter['Phase']], 
                                        edgecolor=None, alpha=0.1) 
    ax.add_patch(rect)
    
ax.set_ylim(-110,30)
ax.grid()

ax2 = axes[0]

ax2.plot(enso['decimal_year'], enso["ENSO"], color='gray', label='ENSO')


for i, index in enumerate(enso_phase_selected.index):
    
    df_iter = enso_phase.loc[index]
    df_iter_next = enso_phase.loc[index+1]

    rect = matplotlib.patches.Rectangle((df_iter['Year'],-5),
                                        1, 10, 
                                        color = phase_color[df_iter['Phase']], 
                                        edgecolor=None, alpha=0.1) 
    ax2.add_patch(rect)

ax2.set_ylim(-2,3)
ax2.set_xlim(1925,2012)
ax2.grid()
ax2.legend()
ax2.set_ylabel("ENSO Index")


fig.savefig("ENSO_plot_sealevel.png")

In [ ]:
phase_color = {'W':'red', 'C':'blue'}
enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<df.index.max().year)&(enso_phase['Phase'] != 'N')]


fig, axes = plt.subplots(2,1, figsize=(8,6), sharex=True, gridspec_kw={'hspace':0})

ax = axes[1]
ax.plot(df['decimal_year'], df["d13c"], ".-",label="raw")
ax.plot(xs, ys, label="spline")
ax.plot(df['decimal_year'], df["MA_d13c"], label="MA w={:}".format(window))
ax.legend(loc='lower left')


for i, index in enumerate(enso_phase_selected.index):
    
    df_iter = enso_phase.loc[index]
    df_iter_next = enso_phase.loc[index+1]

    rect = matplotlib.patches.Rectangle((df_iter['Year'],-5),
                                        1, 10, 
                                        color = phase_color[df_iter['Phase']], 
                                        edgecolor=None, alpha=0.1) 
    ax.add_patch(rect)

ax.set_ylim(-1,0.5)

ax2 = axes[0]

ax2.plot(enso['decimal_year'], enso["ENSO"], color='gray', label='ENSO')

enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<1991)&(enso_phase['Phase'] != 'N')]

for i, index in enumerate(enso_phase_selected.index):
    
    df_iter = enso_phase.loc[index]
    df_iter_next = enso_phase.loc[index+1]

    rect = matplotlib.patches.Rectangle((df_iter['Year'],-5),
                                        1, 10, 
                                        color = phase_color[df_iter['Phase']], 
                                        edgecolor=None, alpha=0.1) 
    ax2.add_patch(rect)

ax2.set_ylim(-2,3)
ax2.set_xlim(1960, 1965)
ax2.grid()
ax2.legend()


In [ ]:
#https://www.isc.ac.uk/cgi-bin/web-db-run?event_id=17289975&out_format=IMS1.0&request=COMPREHENSIVE